In [0]:
%run ./00_config

In [0]:
import requests
from pyspark.sql.functions import current_timestamp

# 1. Carga leve em tempo real (Sem o loop de 90 dias!)
lista_moedas = "bitcoin,ethereum,tether,binancecoin,solana,ripple,cardano,dogecoin,tron,litecoin"
url_api = f"https://api.coingecko.com/api/v3/simple/price?ids={lista_moedas}&vs_currencies=usd&include_24hr_vol=true&include_last_updated_at=true"

try:
    resposta = requests.get(url_api)
    if resposta.status_code == 200:
        dados_json = resposta.json()
        
        registros = []
        for moeda, detalhes in dados_json.items():
            registros.append({
                "moeda_id": moeda,
                "preco_usd": float(detalhes.get("usd", 0.0)),
                "volume_24h": float(detalhes.get("usd_24h_vol", 0.0)),
                "timestamp_api": int(detalhes.get("last_updated_at", 0))
            })

        if registros:
            df_raw = spark.createDataFrame(registros)
            df_bronze = df_raw.withColumn("timestamp_ingestao", current_timestamp())
            caminho_bronze = "abfss://bronze@stcryptolakehouse.dfs.core.windows.net/crypto_prices_raw"
            df_bronze.write.format("delta").mode("append").save(caminho_bronze)
            print(f"Ingestão em tempo real concluída! {len(registros)} registros salvos.")
    else:
        print(f"Erro na API: Status {resposta.status_code}")
except Exception as e:
    print(f"Erro na execução: {str(e)}")

In [0]:
%sql
-- Garante que o catálogo sabe que a Bronze existe
CREATE SCHEMA IF NOT EXISTS hive_metastore.crypto_bronze
LOCATION 'abfss://bronze@stcryptolakehouse.dfs.core.windows.net/';

CREATE EXTERNAL TABLE IF NOT EXISTS hive_metastore.crypto_bronze.crypto_prices_raw
USING DELTA
LOCATION 'abfss://bronze@stcryptolakehouse.dfs.core.windows.net/crypto_prices_raw';